In [ ]:
import numpy
import pandas

import matplotlib.pyplot as plt
plt.style.use('../../mystyle.mplstyle')

import sbruceana

#### Checks on the 1eNp0π pre-selection and selection

The main handle to check for calibration in electron neutrinos is to look into the dE/dx.
The dE/dx does not really provide a fix for the energy scale, but is crucial to cross-check the data/MC discrepancy.
We expect the largest impact from the electron lifetime, which was wildly different between the two cryostats in Run2: comparing data to data at the two cryostats can be used to confirm whether the electron lifetime calibration worked. Any remaining mis-calibration is expected to be arising from the gain, which is known to be tricky especially in the legacy signal processing, and could be absorbed by a scale factor on the calorimetry.

##### Import

In [ ]:
# PATH_TO_SBRUCE = "/Users/triozzi/Analysis/numine/sbruceana/data/cc1e0pi/"

# '''
#   Pre-selection stage, asking that a leading electron exists, without any cut.
#   This is dominated by background below 200 MeV.
# '''

# FILE_CV = "preselection_electron_exists/CNAF_CV_1eNp0pi_NuMI_NoSysts_PreselectionElectronExists.root"

# # off-beam: note that this is data! this needs an explicit
# # calibration assumption, and I'm using the latest one for my own sanity :) 
# FILE_OFFBEAM = "preselection_electron_exists/CNAF_OffBeam_1eNp0pi_NuMI_PreselectionElectronExists_ShowerCaloTools.root"

# # using MC gains (erronously)
# FILE_DATA = "preselection_electron_exists/CNAF_Data_1eNp0pi_NuMI_NoSysts_PreselectionelectronExists.root"

# # updated -- data gains and shower calorimetry tools
# FILE_DATA_CALOTOOLS = "preselection_electron_exists/CNAF_Data_1eNp0pi_NuMI_PreselectionElectronExists_ShowerCaloTools.root"

In [ ]:
PATH_TO_SBRUCE = "/Users/triozzi/Analysis/numine/sbruceana/data/cc1e0pi/"

'''
  Pre-selection stage, asking that a leading electron exists, without any cut.
  This is dominated by background below 200 MeV.
'''

FILE_CV = "preselection_electron/CNAF_CV_1eNp0pi_NuMI_NoSysts_PreselectionElectron.root"

# off-beam: note that this is data! this needs an explicit
# calibration assumption, and I'm using the latest one for my own sanity :) 
FILE_OFFBEAM = "preselection_electron/CNAF_OffBeam_1eNp0pi_NuMI_PreselectionElectron_ShowerCaloTools_EnergyFix.root"

# using MC gains (erronously)
FILE_DATA = "preselection_electron/CNAF_Data_1eNp0pi_NuMI_NoSysts_Preselectionelectron.root"

# updated -- data gains and shower calorimetry tools
FILE_DATA_CALOTOOLS = "preselection_electron/CNAF_Data_1eNp0pi_NuMI_PreselectionElectron_ShowerCaloTools.root"

In [ ]:
# PATH_TO_SBRUCE = "/Users/triozzi/Analysis/nuedis/checks/data/cc1e0pi/"

# '''
#   Pre-selection stage, after cutting on the energy of the leading energy.
# '''

# FILE_CV = "preselection_electron/CNAF_CV_1eNp0pi_NuMI_NoSysts_Preselectionelectron.root"
# FILE_OFFBEAM = ""

# # using data gains from 1D 
# # FILE_DATA = "preselection_electron/CNAF_Data_1eNp0pi_NuMI_NoSysts_Preselectionelectron_CaloFix_2.root"

# # using data gains from 2D
# # FILE_DATA = "preselection_electron/CNAF_Data_1eNp0pi_NuMI_NoSysts_Preselectionelectron_CaloFix.root"

# # using MC gains (erronously)
# FILE_DATA = "preselection_electron/CNAF_Data_1eNp0pi_NuMI_NoSysts_Preselectionelectron.root"

##### Parse data

In [ ]:
# MC
df = sbruceana.io.convert_tree_to_df(
  f"{PATH_TO_SBRUCE}{FILE_CV}",
  "events/selectedNu"  
)
pot = sbruceana.utils.get_POT(f"{PATH_TO_SBRUCE}{FILE_CV}")

df_cos = sbruceana.io.convert_tree_to_df(
  f"{PATH_TO_SBRUCE}{FILE_CV}",
  "events/selectedCos"  
)
pot_cos = sbruceana.utils.get_POT(f"{PATH_TO_SBRUCE}{FILE_CV}")

df_cos['cosmic'] = 1
df = pandas.concat(
  (df, df_cos)
)

# offbeam
df_offbeam = sbruceana.io.convert_tree_to_df(
  f"{PATH_TO_SBRUCE}{FILE_OFFBEAM}",
  "offbeam/selectedOffbeam"  
)
time_offbeam = sbruceana.utils.get_livetime_offbeam(f"{PATH_TO_SBRUCE}{FILE_OFFBEAM}")

# data
df_data = sbruceana.io.convert_tree_to_df(
  f"{PATH_TO_SBRUCE}{FILE_DATA}",
  "events/selectedData"  
)
pot_data = sbruceana.utils.get_POT(f"{PATH_TO_SBRUCE}{FILE_DATA}")
time_data = sbruceana.utils.get_livetime_data(f"{PATH_TO_SBRUCE}{FILE_DATA}")

# data -- calo tools
df_data_calo = sbruceana.io.convert_tree_to_df(
  f"{PATH_TO_SBRUCE}{FILE_DATA_CALOTOOLS}",
  "events/selectedData"  
)
pot_data_calo = sbruceana.utils.get_POT(f"{PATH_TO_SBRUCE}{FILE_DATA_CALOTOOLS}")
time_data_calo = sbruceana.utils.get_livetime_data(f"{PATH_TO_SBRUCE}{FILE_DATA_CALOTOOLS}")

##### Data/MC scale

In [ ]:
IS_AREA_NORMALIZED = True

In [ ]:
fig, ax = plt.subplots(figsize=(4.25, 3.5), layout='constrained')

var = "colldEdx"

# width = 0.3; bins = numpy.arange(0.3, 12+width, width)
width = 0.5; bins = numpy.arange(0.5, 12+width, width)

ax = sbruceana.plotting.plot_by_category_with_offbeam(ax, df, sbruceana.config.CC1E0PI_CATEGORIES, bins, var, df_offbeam, offbeam_scale=time_data_calo/time_offbeam, yscale=pot_data_calo/pot, area_normalized=IS_AREA_NORMALIZED, band=True)

# ax = sbruceana.plotting.plot_data(ax, df_data, bins, var, IS_AREA_NORMALIZED, color='black', label='data')
ax = sbruceana.plotting.plot_data(ax, df_data*1.3, bins, var, IS_AREA_NORMALIZED, color='black', marker='.', label='data', show_xerr=False)
# ax = sbruceana.plotting.plot_data(ax, df_data_calo, bins, var, IS_AREA_NORMALIZED, color='black', label='data (calib.)')

# gfx
ax.set(
  title = f'NuMI MC: {pot:.1e} POT\nRun2 10% NuMI data: {pot_data:.1e} POT',
  xlabel = 'start d$E$/d$x$ [MeV/cm]',
  ylabel = f'slices [a.n.]\n/ {width} MeV/cm',
  xlim   = (bins[0], bins[-1]),
)
leg = ax.legend(fontsize=8.75); leg.get_title().set_fontsize(11)

plt.show()
fig.savefig(f"plots/preselection/preselection_electron_calorimetry_{var}.pdf", dpi=300)

In [ ]:
fig, axes = plt.subplots(figsize=(4.25*2, 4), ncols=2, layout='constrained')

var = "colldEdx"
width = 0.5; bins = numpy.arange(0.5, 12+width, width)

### uncalibrated
ax = axes[0]

ax = sbruceana.plotting.plot_by_category_with_offbeam(ax, df, sbruceana.config.CC1E0PI_CATEGORIES_REDUX, bins, var, df_offbeam, offbeam_scale=time_data_calo/time_offbeam, yscale=pot_data_calo/pot, area_normalized=IS_AREA_NORMALIZED, band=True)
ax = sbruceana.plotting.plot_data(ax, df_data*1.3, bins, var, IS_AREA_NORMALIZED, color='black', label='data', marker='.', show_xerr=False)

# gfx
ax.set(
  title = f'NuMI MC: {pot:.1e} POT\nRun2 10% NuMI data: {pot_data:.1e} POT',
  xlabel = 'start d$E$/d$x$ [MeV/cm]',
  ylabel = f'slices [a.n.]\n/ {width} MeV/cm',
  xlim   = (bins[0], bins[-1]),
  ylim   = (0, 0.29),
)

ax.text(0.5, 1.25, 'Uncalibrated data', transform=ax.transAxes, ha='center', fontsize=20)

### calibrated
ax = axes[1]

ax = sbruceana.plotting.plot_by_category_with_offbeam(ax, df, sbruceana.config.CC1E0PI_CATEGORIES_REDUX, bins, var, df_offbeam, offbeam_scale=time_data_calo/time_offbeam, yscale=pot_data_calo/pot, area_normalized=IS_AREA_NORMALIZED, band=True)
ax = sbruceana.plotting.plot_data(ax, df_data_calo, bins, var, IS_AREA_NORMALIZED, color='black', label='data', marker='.', show_xerr=False)

# gfx
ax.set(
  title = f'NuMI MC: {pot:.1e} POT\nRun2 10% NuMI data: {pot_data:.1e} POT',
  xlabel = 'start d$E$/d$x$ [MeV/cm]',
  # ylabel = f'slices [a.n.]\n/ {width} MeV/cm',
  xlim   = (bins[0], bins[-1]),
  ylim   = (0, 0.29),
)
leg = ax.legend(fontsize=9); leg.get_title().set_fontsize(11)

ax.text(0.5, 1.25, 'Calibrated data', transform=ax.transAxes, ha='center', fontsize=20)

plt.show()
fig.savefig(f"plots/preselection/preselection_electron_calorimetry_{var}_comparison.pdf", dpi=300)

In [ ]:
fig, ax = plt.subplots(figsize=(6.25, 4.75), layout='constrained')

var = "colldEdx"

# width = 0.3; bins = numpy.arange(0.3, 12+width, width)
width = 0.5; bins = numpy.arange(0.5, 10+width, width)

ax = sbruceana.plotting.plot_by_category_with_offbeam(ax, df, sbruceana.config.CC1E0PI_CATEGORIES, bins, var, df_offbeam, offbeam_scale=time_data_calo/time_offbeam, yscale=pot_data_calo/pot, area_normalized=IS_AREA_NORMALIZED, band=True)

# ax = sbruceana.plotting.plot_data(ax, df_data, bins, var, IS_AREA_NORMALIZED, color='black', label='data')
# ax = sbruceana.plotting.plot_data(ax, df_data*1.3, bins, var, IS_AREA_NORMALIZED, color='gray', label='uncalibrated data')

ax = sbruceana.plotting.plot_var_with_offbeam(ax, df_data*1.3, bins, var, df_offbeam, 0, 1, 1, IS_AREA_NORMALIZED, True, color='black', linewidth=1.5, label='uncalibrated data', hatch_style='////')

ax = sbruceana.plotting.plot_data(ax, df_data_calo, bins, var, IS_AREA_NORMALIZED, color='black', label='data', marker='.')

# gfx
ax.set(
  title = f'NuMI MC: {pot:.1e} POT\nRun2 10% NuMI data: {pot_data:.1e} POT',
  xlabel = 'start d$E$/d$x$ [MeV/cm]',
  ylabel = f'slices [a.n.] / {width} MeV/cm',
  xlim   = (bins[0], bins[-1]),
)
leg = ax.legend(fontsize=9.5); leg.get_title().set_fontsize(11)

plt.show()
fig.savefig(f"plots/preselection/preselection_electron_calorimetry_{var}_comparison_v0.pdf", dpi=300)

In [ ]:
fig, axes = plt.subplots(figsize=(4.25*2, 3.8), ncols=2, layout='constrained')

var = "colldEdx"
width = 0.5; bins = numpy.arange(0.5, 10+width, width)

### east/west
ax = axes[0]

ax = sbruceana.plotting.plot_var_with_offbeam(ax, df, bins, var, df_offbeam, offbeam_scale=time_data_calo/time_offbeam, calib_factor=1, weight=1, area_normalized=IS_AREA_NORMALIZED, band=True, color='black', linewidth=1.5, label='MC')

ax = sbruceana.plotting.plot_data(ax, df_data_calo[df_data_calo.vtxx < 0], bins, var, IS_AREA_NORMALIZED, color='C0', label='east', show_xerr=True)
ax = sbruceana.plotting.plot_data(ax, df_data_calo[df_data_calo.vtxx > 0], bins, var, IS_AREA_NORMALIZED, color='C1', label='west', show_xerr=True)

# gfx
ax.set(
  title = f'NuMI MC: {pot:.1e} POT\nRun2 10% NuMI data: {pot_data:.1e} POT',
  xlabel = 'start d$E$/d$x$ [MeV/cm]',
  ylabel = f'slices [a.n.]\n/ {width} MeV/cm',
  xlim   = (bins[0], bins[-1]),
  ylim   = (0, 0.35)
)
leg = ax.legend(fontsize=10); leg.get_title().set_fontsize(11)

### TPC
ax = axes[1]

ax = sbruceana.plotting.plot_var_with_offbeam(ax, df, bins, var, df_offbeam, offbeam_scale=time_data_calo/time_offbeam, calib_factor=1, weight=1, area_normalized=IS_AREA_NORMALIZED, band=True, color='black', linewidth=1.5, label='MC')

ax = sbruceana.plotting.plot_data(ax, df_data_calo[df_data_calo.vtxx < -210], bins, var, IS_AREA_NORMALIZED, color='C0', label='EE', show_xerr=True)
ax = sbruceana.plotting.plot_data(ax, df_data_calo[(df_data_calo.vtxx > -210) & (df_data_calo.vtxx < 0)], bins, var, IS_AREA_NORMALIZED, color='C1', label='EW', show_xerr=True)
ax = sbruceana.plotting.plot_data(ax, df_data_calo[(df_data_calo.vtxx > 0) & (df_data_calo.vtxx < 210)], bins, var, IS_AREA_NORMALIZED, color='C2', label='WE', show_xerr=True)
ax = sbruceana.plotting.plot_data(ax, df_data_calo[df_data_calo.vtxx < 210], bins, var, IS_AREA_NORMALIZED, color='C3', label='WW', show_xerr=True)

# gfx
ax.set(
  title = f'NuMI MC: {pot:.1e} POT\nRun2 10% NuMI data: {pot_data:.1e} POT',
  xlabel = 'start d$E$/d$x$ [MeV/cm]',
  # ylabel = f'slices [a.n.]\n/ {width} MeV/cm',
  xlim   = (bins[0], bins[-1]),
  ylim   = (0, 0.35)
)
leg = ax.legend(fontsize=10); leg.get_title().set_fontsize(11)

fig.suptitle('Calibrated data', fontsize=20)
fig.savefig(f"plots/preselection/preselection_electron_calorimetry_{var}_comparison_cryo_TPC.pdf", dpi=300)

In [ ]:
fig, ax = plt.subplots(figsize=(4.25, 3.5), layout='constrained')

var = "colldEdx"

width = 0.75; bins = numpy.arange(0.25, 9.25+width, width)

ax = sbruceana.plotting.plot_var_with_offbeam(ax, df, bins, var, df_offbeam, offbeam_scale=time_data_calo/time_offbeam, calib_factor=1, weight=1, area_normalized=IS_AREA_NORMALIZED, band=True, color='black', linewidth=1.5, label='MC')

ax = sbruceana.plotting.plot_data(ax, df_data_calo[df_data_calo.vtxx < 0], bins, var, IS_AREA_NORMALIZED, color='C0', label='east', show_xerr=True)
ax = sbruceana.plotting.plot_data(ax, df_data_calo[df_data_calo.vtxx > 0], bins, var, IS_AREA_NORMALIZED, color='C1', label='west', show_xerr=True)

# gfx
ax.set(
  title = f'NuMI MC: {pot:.1e} POT\nRun2 10% NuMI data: {pot_data:.1e} POT',
  xlabel = 'start d$E$/d$x$ [MeV/cm]',
  ylabel = f'slices [a.n.]\n/ {width} MeV/cm',
  xlim   = (bins[0], bins[-1]),
)
leg = ax.legend(fontsize=10); leg.get_title().set_fontsize(11)

plt.show()
fig.savefig(f"plots/preselection/preselection_electron_calorimetry_data_cryo_{var}.pdf", dpi=300)

In [ ]:
fig, ax = plt.subplots(figsize=(4.25, 3.5), layout='constrained')

var = "colldEdx"

width = 0.75; bins = numpy.arange(0.25, 10+width, width)

ax = sbruceana.plotting.plot_var_with_offbeam(ax, df, bins, var, df_offbeam, offbeam_scale=time_data_calo/time_offbeam, calib_factor=1, weight=1, area_normalized=IS_AREA_NORMALIZED, band=True, color='black', linewidth=1.5, label='MC')

ax = sbruceana.plotting.plot_data(ax, df_data_calo[df_data_calo.vtxx < -210], bins, var, IS_AREA_NORMALIZED, color='C0', label='EE', show_xerr=True)
ax = sbruceana.plotting.plot_data(ax, df_data_calo[(df_data_calo.vtxx > -210) & (df_data_calo.vtxx < 0)], bins, var, IS_AREA_NORMALIZED, color='C1', label='EW', show_xerr=True)
ax = sbruceana.plotting.plot_data(ax, df_data_calo[(df_data_calo.vtxx > 0) & (df_data_calo.vtxx < 210)], bins, var, IS_AREA_NORMALIZED, color='C2', label='WE', show_xerr=True)
ax = sbruceana.plotting.plot_data(ax, df_data_calo[df_data_calo.vtxx < 210], bins, var, IS_AREA_NORMALIZED, color='C3', label='WW', show_xerr=True)

# gfx
ax.set(
  title = f'Run2 10% NuMI data: {pot_data:.1e} POT',
  xlabel = 'start d$E$/d$x$ [MeV/cm]',
  ylabel = f'slices [a.n.]\n/ {width} MeV/cm',
  xlim   = (bins[0], bins[-1]),
)
leg = ax.legend(fontsize=10); leg.get_title().set_fontsize(11)

plt.show()
fig.savefig(f"plots/preselection/preselection_electron_calorimetry_data_TPC_{var}.pdf", dpi=300)

In [ ]:
fig, ax = plt.subplots(figsize=(4.25, 3.5), layout='constrained')

var = "colldEdx"

width = 0.5; bins = numpy.arange(0.5, 10+width, width)

ax = sbruceana.plotting.plot_var_with_offbeam(ax, df, bins, var, df_offbeam, offbeam_scale=time_data_calo/time_offbeam, calib_factor=1, weight=1, area_normalized=IS_AREA_NORMALIZED, band=True, color='black', linewidth=1.5, label='MC')

ax = sbruceana.plotting.plot_data(ax, df_data_calo[df_data_calo.vtxx < -210], bins, var, IS_AREA_NORMALIZED, color='red', label='EE')
ax = sbruceana.plotting.plot_data(ax, df_data_calo[df_data_calo.vtxx > -210], bins, var, IS_AREA_NORMALIZED, color='black', label='EW+WE+WW')

# gfx
ax.set(
  title = f'Run2 10% NuMI data: {pot_data:.1e} POT',
  xlabel = 'start d$E$/d$x$ [MeV/cm]',
  ylabel = f'slices [a.n.]\n/ {width} MeV/cm',
  xlim   = (bins[0], bins[-1]),
)
leg = ax.legend(fontsize=10); leg.get_title().set_fontsize(11)

plt.show()
fig.savefig(f"plots/preselection/preselection_electron_calorimetry_data_EE_{var}.pdf", dpi=300)

In [ ]:
def sweep_chi2(
  df,
  df_data,
  var,
  bins,
  fs = numpy.arange(0.6, 1.5, 0.001)
):
  chi_sqs = []
  for f in fs:
    chi_sq, _ = sbruceana.utils.chi2_histograms(df, df_data, var, bins, f)
    chi_sqs.append(chi_sq)

  return fs, chi_sqs

In [ ]:
fig, ax = plt.subplots(figsize=(4.25, 3.5), layout='constrained')

width = 0.2; bins = numpy.arange(0.3, 11+width, width)

var = "ind1dEdx"
fs, chi_sqs = sweep_chi2(df, df_data, var, bins)
min_chi_sq = fs[numpy.argmin(chi_sqs)]
ax.plot(fs, chi_sqs, label=f'Ind-1\n{min_chi_sq:.4f}')

var = "ind2dEdx"
fs, chi_sqs = sweep_chi2(df, df_data, var, bins)
min_chi_sq = fs[numpy.argmin(chi_sqs)]
ax.plot(fs, chi_sqs, label=f'Ind-2\n{min_chi_sq:.4f}')

var = "colldEdx"
fs, chi_sqs = sweep_chi2(df, df_data, var, bins)
min_chi_sq = fs[numpy.argmin(chi_sqs)]
ax.plot(fs, chi_sqs, label=f'Coll\n{min_chi_sq:.4f}')

# gfx
ax.set(
  title = f'NuMI MC: {pot:.1e} POT\nRun2 10% NuMI data: {pot_data:.1e} POT',
  xlabel = 'data scale factor ($\\mathcal{G}=0.0133$)',
  ylabel = 'data/MC $\\chi^2$',
  xlim   = (min(fs), max(fs)),
)
leg = ax.legend(fontsize=10, title='NuMI CV'); leg.get_title().set_fontsize(11)

In [ ]:
# using MC gains
gains = numpy.array([0.01343, 0.01338, 0.01219])

# factors to obtain data/MC match
factors = numpy.array([1, 1, 0.8589])

In [ ]:
# using data gains
gains = numpy.array([0.0133333, 0.0133333, 0.0133333])

# factors to obtain data/MC match
factors = numpy.array([1.1139, 1.1333, 1.219])

In [ ]:
# Collection
0.01219 + (1 - 0.8589) / (1.219 - 0.8589) * (0.01333 - 0.01219)

In [ ]:
# Induction-1
0.01333 + (1 - 1.1139) / 315.9

In [ ]:
# Induction-2
0.01333 + (1 - 1.1333) / 315.9